# Predict Energy Consumption for Smart Grid Optimization

**Use Case:** 48-hour-ahead energy consumption forecasting
**Dataset:** ETT-h1 (Electricity Transformer Temperature) — real hourly data
**API Plan:** Pro

This notebook forecasts 48 hours of energy load and validates accuracy with backtest.


In [ ]:
import os, math, random, requests
import matplotlib.pyplot as plt

BASE_URL = "http://localhost:8000/v1"
HEADERS = {"Content-Type": "application/json", "X-Plan": "pro"}

def load_energy_series():
    cache = os.path.join("..", "..", "benchmarks", ".cache", "ETTh1.csv")
    try:
        import pandas as pd
        df = pd.read_csv(cache)
        series = df["OT"].values[-720:].tolist()
        print("Loaded real ETT-h1 data (last 720 hours = 30 days).")
        return [round(float(v), 4) for v in series]
    except Exception:
        print("Generating synthetic hourly energy series (720 points).")
        rng = random.Random(13)
        series = []
        for i in range(720):
            hour = i % 24
            day = (i // 24) % 7
            base = 8.0
            daily = 2.5 * math.sin(2 * math.pi * hour / 24 - math.pi / 2)
            weekly = 1.0 if day < 5 else -0.5
            noise = rng.gauss(0, 0.3)
            series.append(round(base + daily + weekly + noise, 4))
        return series

series = load_energy_series()
print(f"Series: {len(series)} hourly observations")
print(f"Range: {min(series):.2f} – {max(series):.2f}")


## Step 1 — Forecast 48 Hours Ahead

In [ ]:
response = requests.post(f"{BASE_URL}/forecast/univariate", headers=HEADERS,
    json={"series": series, "horizon": 48, "frequency": "H", "model": "arima",
          "confidence_levels": [0.8, 0.95]})
data = response.json()

mean = data["forecast"]["mean"]
l80  = data["forecast"]["lower_80"]
u80  = data["forecast"]["upper_80"]
l95  = data["forecast"]["lower_95"]
u95  = data["forecast"]["upper_95"]

print(f"Model    : {data['model_used']}")
print(f"Inference: {data['meta']['inference_time_ms']}ms")
print(f"48h forecast: min={min(mean):.2f}, max={max(mean):.2f}, avg={sum(mean)/len(mean):.2f}")


## Step 2 — Hourly Visualization

In [ ]:
n = len(series)
hist_show = min(72, n)
trim = n - hist_show

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor("white")

hist_x = list(range(trim, n))
fore_x = list(range(n - 1, n + 48))
fore_mean = [series[-1]] + mean
fore_l80 = [series[-1]] + l80;  fore_u80 = [series[-1]] + u80
fore_l95 = [series[-1]] + l95;  fore_u95 = [series[-1]] + u95

ax.fill_between(fore_x, fore_l95, fore_u95, alpha=0.12, color="gray", label="95% CI")
ax.fill_between(fore_x, fore_l80, fore_u80, alpha=0.22, color="gray", label="80% CI")
ax.plot(hist_x, series[trim:], color="#2196F3", linewidth=1.5, label="Historical (last 72h)")
ax.plot(fore_x, fore_mean, color="#FF9800", linewidth=2.5, linestyle="--", label="Forecast (48h)")
ax.axvline(x=n - 1, color="#888", linewidth=1, linestyle=":", label="Forecast start")

ax.set_title("Energy Consumption — 48h Ahead Forecast", fontsize=13, fontweight="bold")
ax.set_xlabel("Hour Index"); ax.set_ylabel("Temperature / Load (°C or MW)")
ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
ax.set_facecolor("white")
plt.tight_layout()
plt.savefig("outputs/03_energy_forecast.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()


## Step 3 — Validate with Backtesting

In [ ]:
val_response = requests.post(f"{BASE_URL}/validate", headers=HEADERS,
    json={"series": series[:500], "horizon": 24, "frequency": "H",
          "model": "arima", "n_windows": 3})
val = val_response.json()

metrics = val["backtest_metrics"]
print("=" * 50)
print("Backtest Results (3-window cross-validation)")
print("=" * 50)
print(f"MAE           : {metrics['mae']:.4f}")
print(f"RMSE          : {metrics['rmse']:.4f}")
print(f"MAPE          : {metrics['mape'] * 100:.2f}%")
print(f"Coverage 80%  : {metrics['coverage_80'] * 100:.1f}% (target ≥ 80%)")
print(f"Coverage 95%  : {metrics['coverage_95'] * 100:.1f}% (target ≥ 95%)")
print()
print(f"TSFA achieves {metrics['mape'] * 100:.1f}% MAPE on energy data ✅")
